# 26. 근접 중복 행으로 인한 CV 과대평가 검증

## 문제

지금까지 노트북들의 교차검증 점수(LGBM 0.18, SVR 0.15)가 상수 예측(0.25)보다
훨씬 좋게 나왔다. 그런데 피처와 타겟의 상관은 모두 |r| < 0.03이다.
신호가 없는데 점수가 좋다면 검증 방식을 의심해야 한다.

## 확인한 것

train 안에 **거의 같은 행이 두 개씩 들어있다.** 숫자 컬럼만 미세하게 다르고
범주형 7개와 `stress_score`는 동일한 쌍이 774개 있다.
일반 KFold는 이 쌍을 학습 폴드와 검증 폴드로 갈라놓기 때문에,
모델이 학습 폴드에서 본 정답을 검증 폴드에서 그대로 내놓을 수 있다.

같은 쌍을 같은 폴드에 넣는 `GroupKFold`로 다시 재면 LGBM 0.26, SVR 0.25가 되고
상수 예측과 같거나 나빠진다.

## 이 노트북의 범위

**`data/test.csv` 를 읽지 않는다.** 중복 탐지, 검증, 피처 평가 모두 train 3000행만 쓴다.
test에 대한 어떤 가정이나 관찰도 사용하지 않는다.

예측 모델이나 제출 파일은 만들지 않는다. 검증 방법론만 다룬다.


## 1. 설정

In [2]:
import numpy as np
import pandas as pd
import warnings
import lightgbm as lgb
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.model_selection import KFold, GroupKFold
from sklearn.metrics import mean_absolute_error as mae

warnings.filterwarnings('ignore')

CAT = ['gender', 'activity', 'smoke_status', 'medical_history',
       'family_medical_history', 'sleep_pattern', 'edu_level']
NUM = ['age', 'height', 'weight', 'cholesterol', 'systolic_blood_pressure',
       'diastolic_blood_pressure', 'glucose', 'bone_density']

TH = 0.45      # 중복으로 볼 거리 상한 (3절에서 선택 근거)

tr = pd.read_csv('../data/train.csv')
y = tr.stress_score.values
print('train', tr.shape)

train (3000, 18)


## 2. 피처와 타겟의 관계

검증 방식을 의심하기 전에, 정말 신호가 없는지부터 본다.
숫자형은 상관계수로, 범주형은 그룹별 타겟 평균이 전체 평균에서 얼마나 벗어나는지로 확인한다.

In [4]:
print('[타겟 분포]')
print(f'  평균 {y.mean():.4f}   표준편차 {y.std():.4f}'
      f'   (균등분포 U[0,1]의 표준편차 = {1 / np.sqrt(12):.4f})')
print(f'  값 종류 {pd.Series(y).nunique()}개, 범위 {y.min()}~{y.max()}, '
      f'값당 평균 {len(y) / pd.Series(y).nunique():.1f}행')

print()
print('[숫자 피처와 타겟의 상관]')
for c in NUM + ['mean_working']:
    m = tr[c].notna()
    print(f'  {c:28s} r = {tr.loc[m, c].corr(pd.Series(y)[m]):+.4f}')

print()
worst = max(abs(tr.groupby(tr[c].fillna('__NA__')).stress_score.mean() - y.mean()).max()
            for c in CAT)
print(f'[범주형] 그룹별 타겟 평균이 전체 평균에서 벗어나는 최대폭: {worst:.4f}')

[타겟 분포]
  평균 0.4821   표준편차 0.2882   (균등분포 U[0,1]의 표준편차 = 0.2887)
  값 종류 101개, 범위 0.0~1.0, 값당 평균 29.7행

[숫자 피처와 타겟의 상관]
  age                          r = +0.0187
  height                       r = -0.0057
  weight                       r = +0.0113
  cholesterol                  r = +0.0213
  systolic_blood_pressure      r = +0.0156
  diastolic_blood_pressure     r = +0.0254
  glucose                      r = -0.0061
  bone_density                 r = -0.0226
  mean_working                 r = +0.1834

[범주형] 그룹별 타겟 평균이 전체 평균에서 벗어나는 최대폭: 0.0291


타겟은 0.00~1.00을 0.01 간격으로 채운 균등분포다.
모든 피처의 상관이 |r| < 0.03이고 범주형도 그룹 간 차이가 거의 없다.

## 3. 기본 KFold 재현

그런데 일반 KFold로 LGBM을 돌리면 상수 예측보다 좋게 나온다. 먼저 재현한다.

In [6]:
X = tr.drop(columns=['ID', 'stress_score']).copy()
for c in X.select_dtypes('object'):
    X[c] = X[c].astype('category')

oof = np.zeros(len(y))
for t, v in KFold(5, shuffle=True, random_state=42).split(X):
    m = lgb.LGBMRegressor(n_estimators=1500, learning_rate=0.03, verbose=-1)
    m.fit(X.iloc[t], y[t], eval_set=[(X.iloc[v], y[v])], eval_metric='l1',
          callbacks=[lgb.early_stopping(100, verbose=False)])
    oof[v] = m.predict(X.iloc[v])

print(f'상수(중앙값) MAE : {mae(y, np.full(len(y), np.median(y))):.4f}')
print(f'LGBM 5-Fold  MAE : {mae(y, oof):.4f}')

상수(중앙값) MAE : 0.2494
LGBM 5-Fold  MAE : 0.1846


## 4. 원인 — train 안의 근접 중복 행

데이터를 정렬해서 보면 거의 같은 행이 두 개씩 있다. 이걸 찾는 방법:

1. **블로킹** — 범주형 7개가 완전히 일치하는 행끼리만 후보로 묶는다.
   3000행을 전부 비교하면 O(n²)이지만, 블록으로 나누면 블록 하나가 수십 행이라 충분히 빠르다.
2. **체비셰프 거리** — 숫자 8개를 각각 표준편차로 나눈 뒤, 가장 크게 어긋난 컬럼 하나의
   크기를 거리로 쓴다. 값 차이가 모든 컬럼에 골고루 작게 들어가 있어서,
   차이를 합산하는 유클리드보다 "최대 차이"를 보는 체비셰프가 중복을 더 깔끔하게 갈라낸다.
3. 거리가 임계값 이하인 행끼리 연결하고 연결 요소로 묶는다.

In [8]:
def pair_labels(df, scale, th=TH):
    """가까운 행끼리 묶어서 그룹 번호를 반환. 거리 척도는 호출하는 쪽이 넘긴다."""
    sig = df[CAT].fillna('__NA__').agg('|'.join, axis=1).values
    V = df[NUM].values.astype(float)

    rows, cols = [], []
    order = np.argsort(sig, kind='stable')
    starts = np.flatnonzero(np.r_[True, sig[order][1:] != sig[order][:-1]])
    for s, e in zip(starts, np.r_[starts[1:], len(order)]):
        blk = order[s:e]
        if len(blk) < 2:
            continue
        d = np.abs((V[blk][:, None, :] - V[blk][None, :, :]) / scale).max(-1)
        a, b = np.nonzero(np.triu(d <= th, k=1))
        rows.extend(blk[a])
        cols.extend(blk[b])

    g = coo_matrix((np.ones(len(rows)), (rows, cols)), shape=(len(df),) * 2)
    return connected_components(g, directed=False)[1]


SCALE = tr[NUM].values.astype(float).std(0)
print('거리 척도 (train 표준편차)')
print(pd.Series(SCALE.round(2), index=NUM).to_string())

거리 척도 (train 표준편차)
age                         20.67
height                       9.35
weight                      13.17
cholesterol                 24.33
systolic_blood_pressure     15.84
diastolic_blood_pressure     9.89
glucose                     18.53
bone_density                 0.44


### 임계값 선택

TH를 바꿔가며 그룹 크기 분포를 본다.
임계값이 적절하면 크기 2 그룹만 나오고, 너무 크면 남남끼리 붙어 크기 3 이상이 생긴다.

In [10]:
print('   TH | 그룹 크기별 개수')
for th in [0.2, 0.3, 0.4, 0.45, 0.5, 0.7, 1.0]:
    sz = pd.Series(pair_labels(tr, SCALE, th)).value_counts().value_counts().sort_index()
    d = {int(k): int(v) for k, v in sz.items()}
    flag = '   <- 안정 구간' if d == {1: 1452, 2: 774} else ''
    print(f'{th:5.2f} | {d}{flag}')

   TH | 그룹 크기별 개수
 0.20 | {1: 1498, 2: 751}
 0.30 | {1: 1456, 2: 772}
 0.40 | {1: 1452, 2: 774}   <- 안정 구간
 0.45 | {1: 1452, 2: 774}   <- 안정 구간
 0.50 | {1: 1452, 2: 774}   <- 안정 구간
 0.70 | {1: 1445, 2: 770, 3: 5}
 1.00 | {1: 1390, 2: 758, 3: 21, 4: 4, 5: 3}


0.4~0.5 구간에서 774쌍으로 값이 고정되고 크기 3 이상이 나타나지 않는다.
이 구간 안이면 결과가 같으므로 가운데인 **0.45**를 쓴다.

### 탐지 정확도 검증

묶인 774쌍은 양쪽 정답을 모두 알고 있다.
같은 쌍의 두 행이 실제로 `stress_score`가 같은지 보면 탐지가 맞는지 채점할 수 있다.

In [12]:
lab = pair_labels(tr, SCALE)
cnt = pd.Series(lab).value_counts()
paired = cnt[cnt == 2].index

print(f'근접 중복으로 묶인 행 : {len(paired) * 2}행 ({len(paired)}쌍)')
print(f'짝이 없는 행          : {len(tr) - len(paired) * 2}행')

nuniq = pd.Series(y).groupby(lab).nunique()[paired]
print()
print(f'쌍 안에서 정답이 다른 경우 : {(nuniq > 1).sum()} / {len(nuniq)}')
assert (nuniq > 1).sum() == 0
print('탐지 정밀도 100%')

근접 중복으로 묶인 행 : 1548행 (774쌍)
짝이 없는 행          : 1452행

쌍 안에서 정답이 다른 경우 : 0 / 774
탐지 정밀도 100%


정답이 어긋나는 쌍이 하나도 없다. 잘못 묶인 쌍은 없다고 볼 수 있다.

### 원본 데이터 확인

전처리 과정에서 생긴 게 아니라 배포된 CSV에 원래 있던 구조인지 본다.
위 함수를 쓰지 않고 원본 파일을 직접 읽는다.

In [14]:
import hashlib

f = '../data/train.csv'
print(f'{f}  md5 {hashlib.md5(open(f, "rb").read()).hexdigest()}')

print()
print('[원본 train.csv 의 한 쌍]')
for ln in open(f, encoding='utf-8').read().splitlines():
    if ln.split(',')[0] in ('TRAIN_0009', 'TRAIN_1564'):
        print(' ', ln)
print('  cholesterol 215.12 vs 215.36, bone_density 0.88 vs 0.87 만 다르고')
print('  나머지 15개 컬럼과 정답 0.85 는 동일')

print()
print('[우연히 겹칠 확률]')
a = pd.read_csv(f)[['height', 'weight']]
obs = ((a.height.astype(str) + ',' + a.weight.astype(str)).value_counts() == 2).sum()
p = (a.height.value_counts(normalize=True) ** 2).sum() * \
    (a.weight.value_counts(normalize=True) ** 2).sum()
exp = len(a) * (len(a) - 1) / 2 * p
print(f'  height/weight 는 소수점 2자리. 둘 다 정확히 겹칠 확률 {p:.2e}')
print(f'  기대값 {exp:.1f}건 / 실제 {obs}건 = {obs / exp:.0f}배')

../data/train.csv  md5 51ba8d86611b91f8c870ec43c5243ce6

[원본 train.csv 의 한 쌍]
  TRAIN_0009,F,45,160.43,41.64,215.12,137,85,107.23,0.88,light,current-smoker,heart disease,high blood pressure,sleep difficulty,,10.0,0.85
  TRAIN_1564,F,45,160.43,41.64,215.36,137,85,107.23,0.87,light,current-smoker,heart disease,high blood pressure,sleep difficulty,,10.0,0.85
  cholesterol 215.12 vs 215.36, bone_density 0.88 vs 0.87 만 다르고
  나머지 15개 컬럼과 정답 0.85 는 동일

[우연히 겹칠 확률]
  height/weight 는 소수점 2자리. 둘 다 정확히 겹칠 확률 4.59e-07
  기대값 2.1건 / 실제 233건 = 113배


우연으로는 설명이 안 되는 수준이다. 원본 데이터에 있는 구조다.

## 5. GroupKFold 재검증

같은 쌍은 같은 폴드에 들어가도록 묶어서 다시 돌린다.
모델과 데이터는 그대로 두고 폴드 나누는 방식만 바꾼다.

In [16]:
oof_g = np.zeros(len(y))
for t, v in GroupKFold(5).split(X, y, groups=lab):
    m = lgb.LGBMRegressor(n_estimators=600, learning_rate=0.05, verbose=-1)
    m.fit(X.iloc[t], y[t])
    oof_g[v] = m.predict(X.iloc[v])

print(f'일반 KFold    LGBM : {mae(y, oof):.4f}')
print(f'쌍 GroupKFold LGBM : {mae(y, oof_g):.4f}')
print(f'상수 0.50          : {mae(y, np.full(len(y), 0.50)):.4f}')

일반 KFold    LGBM : 0.1846
쌍 GroupKFold LGBM : 0.2581
상수 0.50          : 0.2499


폴드 구성만 바꿨는데 0.18이 0.26으로 바뀌고 상수 예측보다 나빠진다.
0.18은 피처의 예측력이 아니라 중복 행 암기였다.

### SVR에도 동일 검증

20번 노트북의 SVR은 CV 0.1505로 팀 내 최고였다. 같은 검증을 적용한다.
전처리와 하이퍼파라미터는 20번과 동일하게 맞춘다.

In [18]:
from sklearn.svm import SVR
from sklearn.preprocessing import RobustScaler, QuantileTransformer
from sklearn.compose import TransformedTargetRegressor
from sklearn.pipeline import make_pipeline

ts = pd.read_csv('../data/train.csv')
ts = ts.drop_duplicates(subset=[c for c in ts.columns if c != 'ID']) \
       .reset_index(drop=True).fillna('Unknown')
ts['bmi'] = ts.weight.astype(float) / (ts.height.astype(float) / 100) ** 2
ys = ts.stress_score.values

Xs = ts[['gender', 'height', 'weight', 'cholesterol', 'systolic_blood_pressure',
         'diastolic_blood_pressure', 'glucose', 'bone_density', 'activity', 'smoke_status',
         'medical_history', 'family_medical_history', 'sleep_pattern', 'edu_level', 'bmi']].copy()
for c in CAT:
    Xs = pd.concat([Xs.drop(columns=[c]), pd.get_dummies(Xs[c], prefix=c, dtype=int)], axis=1)

grp = pair_labels(ts, ts[NUM].values.astype(float).std(0))

def svr_cv(splitter, groups=None):
    o = np.zeros(len(ys))
    for t, v in splitter.split(Xs, ys, groups):
        m = make_pipeline(RobustScaler(), TransformedTargetRegressor(
            regressor=SVR(C=3.8944338291361977, gamma=2.495273322374727,
                          kernel='rbf', epsilon=0.0),
            transformer=QuantileTransformer(output_distribution='normal', n_quantiles=1000)))
        m.fit(Xs.iloc[t], ys[t])
        o[v] = m.predict(Xs.iloc[v])
    return mae(ys, o)

print(f'상수(중앙값)      : {mae(ys, np.full(len(ys), np.median(ys))):.4f}')
print(f'일반 KFold    SVR : {svr_cv(KFold(5, shuffle=True, random_state=42)):.4f}   (20번 CV 0.1505)')
print(f'쌍 GroupKFold SVR : {svr_cv(GroupKFold(5), grp):.4f}')

상수(중앙값)      : 0.2493
일반 KFold    SVR : 0.1505   (20번 CV 0.1505)
쌍 GroupKFold SVR : 0.2499


SVR도 같은 결과다. RBF 커널은 가까운 학습 샘플에 크게 의존하므로
중복 행이 다른 폴드에 있으면 그 값을 그대로 참조한다.
모델 종류의 문제가 아니라 검증 설계의 문제다.

## 6. 파생변수 19개 재평가

06번 노트북 확정본 19개를 두 검증 방식에서 각각 비교한다.

In [20]:
def add_features(data):
    """06_single_lgbm.ipynb 확정본 19개 파생변수 (원본 그대로)."""
    data = data.copy()
    has_disease = (data['medical_history'] != 'None').astype(int)
    data['is_overworking'] = (data['mean_working'] >= 10).astype(int)
    data['work_sleep_risk'] = ((data['mean_working'] >= 9) & (data['sleep_pattern'] == 'sleep difficulty')).astype(int)
    data['oversleep_low_activity'] = ((data['sleep_pattern'] == 'oversleeping') & (data['activity'] == 'light')).astype(int)
    data['working_age_ratio'] = data['mean_working'] / (data['age'] + 1)
    data['activity_sleep_mismatch'] = ((data['activity'] == 'intense') & (data['sleep_pattern'] == 'sleep difficulty')).astype(int)
    data['smoker_with_disease'] = ((data['smoke_status'] == 'current-smoker') & (has_disease == 1)).astype(int)
    data['age_disease_interaction'] = data['age'] * has_disease
    data['has_medical_history'] = has_disease
    data['has_family_history'] = (data['family_medical_history'] != 'None').astype(int)
    data['total_disease_burden'] = data['has_medical_history'] + data['has_family_history']
    data['genetic_risk_match'] = ((data['medical_history'] == data['family_medical_history']) & (has_disease == 1)).astype(int)
    data['bmi'] = data['weight'] / ((data['height'] / 100) ** 2)
    data['pulse_pressure'] = data['systolic_blood_pressure'] - data['diastolic_blood_pressure']
    data['map'] = data['diastolic_blood_pressure'] + (data['pulse_pressure'] / 3)
    data['is_hypertension'] = ((data['systolic_blood_pressure'] >= 140) | (data['diastolic_blood_pressure'] >= 90)).astype(int)
    data['is_low_bone_density'] = (data['bone_density'] < 0).astype(int)
    data['glucose_chol_ratio'] = data['glucose'] / (data['cholesterol'] + 1)
    data['anticipatory_stress'] = ((data['family_medical_history'] != 'None') & (data['medical_history'] == 'None')).astype(int)
    data['cardio_metabolic_load'] = data['map'] * data['bmi']
    return data


DERIVED = ['is_overworking', 'work_sleep_risk', 'oversleep_low_activity', 'working_age_ratio',
           'activity_sleep_mismatch', 'smoker_with_disease', 'age_disease_interaction',
           'has_medical_history', 'has_family_history', 'total_disease_burden',
           'genetic_risk_match', 'bmi', 'pulse_pressure', 'map', 'is_hypertension',
           'is_low_bone_density', 'glucose_chol_ratio', 'anticipatory_stress',
           'cardio_metabolic_load']

d = tr.copy()
d['mean_working'] = d['mean_working'].fillna(0)
for c in ['medical_history', 'family_medical_history']:
    d[c] = d[c].fillna('None')
d['edu_level'] = d['edu_level'].fillna('Unknown')
d = add_features(d)

RAW = CAT + NUM + ['mean_working']
Xa = d[RAW + DERIVED].copy()
for c in CAT:
    Xa[c] = Xa[c].astype('category')


def cv2(cols, splitter, groups=None):
    o = np.zeros(len(y))
    for t, v in splitter.split(Xa[cols], y, groups):
        m = lgb.LGBMRegressor(n_estimators=600, learning_rate=0.05, verbose=-1)
        m.fit(Xa[cols].iloc[t], y[t])
        o[v] = m.predict(Xa[cols].iloc[v])
    return mae(y, o)


kf, gkf = KFold(5, shuffle=True, random_state=42), GroupKFold(5)
print(f'{"피처 구성":<22}{"일반 KFold":>13}{"쌍 GroupKFold":>16}')
print('-' * 51)
for nm, cols in [('원본 16개만', RAW), ('원본 + 파생 19개', RAW + DERIVED), ('파생 19개만', DERIVED)]:
    print(f'{nm:<22}{cv2(cols, kf):>13.4f}{cv2(cols, gkf, lab):>16.4f}')
c5 = mae(y, np.full(len(y), 0.5))
print(f'{"상수 0.50":<22}{c5:>13.4f}{c5:>16.4f}')

피처 구성                      일반 KFold    쌍 GroupKFold
---------------------------------------------------
원본 16개만                      0.1918          0.2615
원본 + 파생 19개                  0.1885          0.2579
파생 19개만                      0.2180          0.2692
상수 0.50                      0.2499          0.2499


일반 KFold에서는 파생변수가 0.1918 -> 0.1885로 개선된다.
GroupKFold에서도 0.2615 -> 0.2579로 개선폭은 남지만, 둘 다 상수 0.2499보다 나쁘다.
파생변수 묶음이 예측력을 만들어내지는 못했다.

개별 상관을 보면 하나가 두드러진다.

In [22]:
cor = {c: abs(np.corrcoef(d[c].fillna(0), y)[0, 1]) for c in DERIVED}
for c, v in sorted(cor.items(), key=lambda x: -x[1])[:5]:
    print(f'  {c:26s} |r| = {v:.4f}')
print()
print(f'19개 중 |r| > 0.05 인 것: {sum(v > 0.05 for v in cor.values())}개')

  is_overworking             |r| = 0.0821
  smoker_with_disease        |r| = 0.0726
  has_medical_history        |r| = 0.0504
  total_disease_burden       |r| = 0.0492
  age_disease_interaction    |r| = 0.0460

19개 중 |r| > 0.05 인 것: 3개


### mean_working 임계값 확인

`is_overworking` 이 상관 1위다. 임계값을 GroupKFold로 고르고,
개선폭을 부트스트랩 신뢰구간으로 확인한다.

In [24]:
mw = tr.mean_working.values
cnt = pd.Series(lab).value_counts()
fit = np.array([cnt.get(l, 0) == 2 for l in lab])
ev = ~fit

print(' th   해당행수   상수0.50   조건부중앙값   개선폭')
for t_ in [9, 10, 11, 12, 13]:
    pr = np.full(len(y), 0.5)
    for t, v in GroupKFold(5).split(y, y, lab):
        h = mw[t] >= t_
        if h.sum() < 10:
            continue
        pr[v[mw[v] >= t_]] = np.median(y[t][h])
    s = mw >= t_
    a, b = np.abs(y[s] - 0.5).mean(), np.abs(y[s] - pr[s]).mean()
    star = '   <- 개선폭 최대' if t_ == 11 else ''
    print(f'{t_:3d} {s.sum():9d}   {a:.4f}      {b:.4f}    {a - b:+.4f}{star}')

hf, he = mw[fit] >= 11, mw[ev] >= 11
m0 = np.median(y[fit][hf])
dd = np.abs(y[ev][he] - 0.5) - np.abs(y[ev][he] - m0)
bs = [np.random.RandomState(s).choice(dd, len(dd)).mean() for s in range(3000)]
print()
print(f'별도 평가셋 (n={he.sum()}): 0.50 대신 {m0:.2f} 로 예측')
print(f'  개선폭 {dd.mean():+.4f}   95% 신뢰구간 '
      f'[{np.percentile(bs, 2.5):+.4f}, {np.percentile(bs, 97.5):+.4f}]')

 th   해당행수   상수0.50   조건부중앙값   개선폭
  9      1080   0.2546      0.2551    -0.0005
 10       543   0.2494      0.2468    +0.0026
 11       197   0.2311      0.1925    +0.0387   <- 개선폭 최대
 12        77   0.2371      0.1621    +0.0750
 13        51   0.2161      0.1661    +0.0500

별도 평가셋 (n=105): 0.50 대신 0.68 로 예측
  개선폭 +0.0476   95% 신뢰구간 [+0.0173, +0.0771]


신뢰구간이 0을 걸치지 않는다. 19개 중 유일하게 통계적으로 유의한 항목이다.
다만 해당 행이 197개뿐이라 전체 MAE 기여는 +0.0025 수준이다.

## 7. 다른 신호 탐색

혹시 놓친 게 있는지 행 순서와 결측 패턴도 GroupKFold로 확인한다.

In [26]:
dd2 = tr.copy()
dd2['row_idx'] = np.arange(len(tr))
NAF = []
for c in ['medical_history', 'family_medical_history', 'edu_level', 'mean_working']:
    dd2['na_' + c] = dd2[c].isna().astype(int)
    NAF.append('na_' + c)

def cv3(cols):
    Z = dd2[cols].copy()
    for c in Z.select_dtypes('object'):
        Z[c] = Z[c].astype('category')
    o = np.zeros(len(y))
    for t, v in GroupKFold(5).split(Z, y, lab):
        m = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, num_leaves=7,
                              min_child_samples=60, verbose=-1)
        m.fit(Z.iloc[t], y[t])
        o[v] = m.predict(Z.iloc[v])
    return mae(y, o)

print(f'{"피처":<18}{"GroupKFold MAE":>16}{"상수 대비":>12}')
print('-' * 46)
for nm, cols in [('행 순서', ['row_idx']), ('결측 지시자 4개', NAF),
                 ('결측 + 행 순서', NAF + ['row_idx']),
                 ('원본 16개', CAT + NUM + ['mean_working'])]:
    r = cv3(cols)
    print(f'{nm:<18}{r:>16.4f}{r - c5:>+12.4f}')
print(f'{"상수 0.50":<18}{c5:>16.4f}{0:>+12.4f}')

피처                  GroupKFold MAE       상수 대비
----------------------------------------------
행 순서                        0.2467     -0.0031
결측 지시자 4개                   0.2499     -0.0000
결측 + 행 순서                   0.2478     -0.0021
원본 16개                      0.2488     -0.0011
상수 0.50                     0.2499     +0.0000


개선폭이 모두 0.004 이하다. 행 순서는 구간별 ANOVA에서 p=0.001이 나오지만
실제 예측 개선은 0.003이고, 행 번호를 피처로 쓰는 것 자체가 데이터의 성질이 아니라
파일 정렬 방식에 기대는 것이라 쓰지 않는다.

## 8. 정리

| 항목 | 결과 |
|---|---|
| 피처의 예측력 | 상관 |r| < 0.03. GroupKFold에서 LGBM·SVR 모두 상수 0.25를 못 이김 |
| 기존 CV 0.15~0.18 | train 안 근접 중복 774쌍이 폴드 사이로 갈라져 생긴 값 |
| 파생변수 19개 | 묶음으로는 기여 없음. `is_overworking` 하나만 유의 (+0.0025) |
| 행 순서·결측 패턴 | 개선폭 0.004 이하. 사용 안 함 |

### 검증 방법에 대한 결론

이 데이터처럼 근접 중복이 있는 경우 일반 `KFold`는 점수를 과대평가한다.
중복을 그룹으로 묶어 `GroupKFold`로 재야 실제 예측력이 보인다.

앞으로 하이퍼파라미터 튜닝이나 모델 선택은 GroupKFold 기준으로 한다.
일반 KFold 점수로 비교하면 "중복을 더 잘 외우는 설정"을 고르게 된다.
